<a href="https://colab.research.google.com/github/JuliMiralles/Diplomatura-en-An-lisis-Cuantitativo-y-Machine-Learning-aplicado-al-Riesgo-Financiero/blob/main/CursoVAR_GARCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

APLICACION — ARCH/GARCH y Riesgo Financiero

In [ ]:
!pip install arch

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import yfinance as yf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from arch import arch_model

In [ ]:
# DATOS
prices = yf.download("^GSPC", start="2010-01-01", end="2026-01-01",
                     auto_adjust=True, progress=False)["Close"].squeeze().dropna()
returns = 100*np.log(prices/prices.shift(1)).dropna()

In [ ]:
# 1. EXPLORACIÓN
# Objetivo: reconocer las caracteristicas empiricas de los retornos financieros
print(returns.describe())
print("Asimetría:", returns.skew())
print("Curtosis:", returns.kurtosis()+3)

fig, ax = plt.subplots(2,1,figsize=(12,7),sharex=True)
ax[0].plot(returns); ax[0].set_title("Retornos")
ax[1].plot(returns**2); ax[1].set_title("Retornos al cuadrado")
plt.tight_layout(); plt.show()

plot_acf(returns,lags=30); plt.title("ACF de retornos"); plt.show()
plot_acf(returns**2,lags=30); plt.title("ACF de retornos²"); plt.show()

In [ ]:
# 2. ARIMA Y RESIDUOS
# Objetivo: distinguir el modelado de la media condicional del modelado de la variancia condicional
arima = ARIMA(returns,order=(3,0,3)).fit()
print(arima.summary())
resid = pd.Series(arima.resid,index=returns.index).dropna()
print("Ljung-Box residuos:")
print(acorr_ljungbox(resid,lags=[10,20],return_df=True))
print("Ljung-Box residuos²:")
print(acorr_ljungbox(resid**2,lags=[10,20],return_df=True))

In [ ]:
# 3. ARCH(5)
# Objetivo: detectar homocedasticidad condicional y estimar un modelo ARCH
arch5 = arch_model(returns,mean="Constant",vol="ARCH",p=5,dist="normal").fit(disp="off")
print(arch5.summary())
lm, lm_p, f, f_p = het_arch(resid,nlags=10)
print("ARCH test:", {"LM":lm,"LM p-value":lm_p,"F":f,"F p-value":f_p})

In [ ]:
# 4. GARCH(1,1)
# Objetivo: interpretar GARCH como una forma parsimoniosa de representar persistencia en la volatilidad
garch = arch_model(returns,mean="Constant",vol="GARCH",p=1,q=1,dist="normal").fit(disp="off")
print(garch.summary())
a,b,w = garch.params["alpha[1]"],garch.params["beta[1]"],garch.params["omega"]
print("alpha+beta =",a+b)
if a+b < 1:
    print("Varianza de largo plazo =",w/(1-a-b))

garch.plot(); plt.show()

In [ ]:
# 5. GARCH-NORMAL VS STUDENT-t
# Objetivo: analizar la importancia de la distribucion de los shocks
gn = arch_model(returns,mean="Constant",vol="GARCH",p=1,q=1,dist="normal").fit(disp="off")
gt = arch_model(returns,mean="Constant",vol="GARCH",p=1,q=1,dist="t").fit(disp="off")
print(pd.DataFrame({
    "Normal":[gn.loglikelihood,gn.aic,gn.bic],
    "Student-t":[gt.loglikelihood,gt.aic,gt.bic]},
    index=["LogLik","AIC","BIC"]))
z = pd.Series(gt.std_resid).dropna()
print(acorr_ljungbox(z,lags=[10],return_df=True))
print(acorr_ljungbox(z**2,lags=[10],return_df=True))

In [ ]:
# 6. FORECAST 1, 5 Y 10 DÍAS
# Objetivo: transformar las estimaciones del modelo en pronosticos utiles para gestion de riesgo
fc = gt.forecast(horizon=10,reindex=False)
vf = fc.variance.iloc[-1]
print("Forecast de varianza:")
print(vf)
for h in [1,5,10]:
    print(h,"días: volatilidad acumulada ≈",np.sqrt(vf.iloc[:h].sum()),"%")

In [ ]:
# 7. VaR: train/test
# Objetivo: construir una medida de riesgo fuera de muestra evitando look-ahead bias
split="2023-01-01"
train=returns.loc[:split].copy()
test=returns.loc[split:].copy()

def rolling_garch_var(train,test,window=1000,alpha=.01,dist="t"):
    history=train.copy(); out=[]
    for date,r in test.items():
        sample=history.iloc[-window:]
        m=arch_model(sample,mean="Constant",vol="GARCH",p=1,q=1,dist=dist).fit(disp="off")
        f=m.forecast(horizon=1,reindex=False)
        mu=m.params["mu"]; sigma=np.sqrt(f.variance.iloc[-1,0])
        if dist=="normal":
            q=stats.norm.ppf(alpha)
        else:
            nu=m.params["nu"]
            q=stats.t.ppf(alpha,df=nu)*np.sqrt((nu-2)/nu)
        out.append(-(mu+sigma*q))
        history=pd.concat([history,pd.Series([r],index=[date])])
    return pd.Series(out,index=test.index,name="VaR")

def rolling_hist_var(train,test,window=1000,alpha=.01):
    history=train.copy(); out=[]
    for date,r in test.items():
        sample=history.iloc[-window:]
        out.append(-sample.quantile(alpha))
        history=pd.concat([history,pd.Series([r],index=[date])])
    return pd.Series(out,index=test.index,name="VaR")

var99_hist=rolling_hist_var(train,test,1000,.01)
var95_hist=rolling_hist_var(train,test,1000,.05)
var99_norm=rolling_garch_var(train,test,1000,.01,"normal")
var99_t=rolling_garch_var(train,test,1000,.01,"t")

In [ ]:
# 8. BACKTESTING
# Objetivo: evaluar si el VaR cumple con la frecuencia y estructura temporal del excepciones esperadas
def backtest(returns,var):
    d=pd.concat([returns.rename("r"),var.rename("VaR")],axis=1).dropna()
    d["exception"]=d["r"] < -d["VaR"]
    return d, {"n":len(d),"exceptions":int(d.exception.sum()),
               "rate":d.exception.mean()}

results={}
for name,v in {"Historical":var99_hist,"GARCH-Normal":var99_norm,
               "GARCH-Student-t":var99_t}.items():
    d,s=backtest(test,v); results[name]=s
print(pd.DataFrame(results).T.assign(expected_rate=.01))

In [ ]:
# KUPIEC
def kupiec(x,n,alpha=.01):
    phat=np.clip(x/n,1e-12,1-1e-12)
    lr=-2*((n-x)*np.log(1-alpha)+x*np.log(alpha)
           -(n-x)*np.log(1-phat)-x*np.log(phat))
    return lr,1-stats.chi2.cdf(lr,1)

for name,s in results.items():
    print(name,"Kupiec:",kupiec(s["exceptions"],s["n"]))

In [ ]:
# CHRISTOFFERSEN: INDEPENDENCIA
def christoffersen(ex):
    x=np.asarray(ex,dtype=int)
    n00=n01=n10=n11=0
    for i in range(1,len(x)):
        if x[i-1]==0 and x[i]==0:n00+=1
        elif x[i-1]==0 and x[i]==1:n01+=1
        elif x[i-1]==1 and x[i]==0:n10+=1
        else:n11+=1
    def ratio(a,b): return a/b if b else 0
    p01=np.clip(ratio(n01,n00+n01),1e-12,1-1e-12)
    p11=np.clip(ratio(n11,n10+n11),1e-12,1-1e-12)
    p=np.clip(ratio(n01+n11,n00+n01+n10+n11),1e-12,1-1e-12)
    l1=n00*np.log(1-p01)+n01*np.log(p01)+n10*np.log(1-p11)+n11*np.log(p11)
    l0=(n00+n10)*np.log(1-p)+(n01+n11)*np.log(p)
    lr=-2*(l0-l1)
    return lr,1-stats.chi2.cdf(lr,1)

for name,v in {"Historical":var99_hist,"GARCH-Normal":var99_norm,
               "GARCH-Student-t":var99_t}.items():
    d,_=backtest(test,v)
    print(name,"Christoffersen independence:",christoffersen(d.exception))

In [ ]:
# GRÁFICO
fig,ax=plt.subplots(figsize=(14,6))
ax.plot(test,label="Retorno")
ax.plot(var99_hist.index,-var99_hist,label="VaR Histórico 99%")
ax.plot(var99_norm.index,-var99_norm,label="GARCH-Normal 99%")
ax.plot(var99_t.index,-var99_t,label="GARCH-Student-t 99%")
ax.axhline(0,linewidth=.8); ax.legend(); ax.set_title("Backtesting VaR")
plt.tight_layout(); plt.show()

In [ ]:
# 9. CASO: POSICIÓN USD 10 MILLONES
# Objetivo: traducir una medida estadistica del riesgo a una magnitud economica
position=10_000_000
last_var_pct=var99_t.dropna().iloc[-1]
print("VaR porcentual:",last_var_pct,"%")
print("VaR monetario: USD",position*last_var_pct/100)